# 5. Hannah Model Metrics Extraction

This notebook extracts a compact set of comparable metrics (data misfit, target volume fraction, and report length) for GPT, QWEN, Gemini, and Claude.


## Block 1: Define Parsing Utilities

Load model paths and helper parsers for inversion logs, geology volumes, and report text statistics.


In [1]:
# -------------------------
# Block 1: Define Parsing Utilities
# -------------------------
import ast
import re
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()

MODEL_DIRS = {
    "GPT": ROOT / "Hannah_Inversion_GPT",
    "QWEN": ROOT / "Hannah_Inversion_Qwen",
    "Gemini": ROOT / "Hannah_Inversion_gemini",
    "Claude": ROOT / "Hannah_Inversion_claude",
}


def _latest_file(folder: Path, pattern: str):
    files = sorted(folder.glob(pattern), key=lambda p: p.stat().st_mtime)
    return files[-1] if files else None


def parse_data_misfit(log_path: Path) -> dict:
    """Extract only final gravity/magnetic data-misfit terms from Output_*.txt."""
    if log_path is None or (not log_path.exists()):
        return {}

    lines = [ln.strip() for ln in log_path.read_text(encoding="utf-8", errors="ignore").splitlines() if ln.strip()]
    data_lines = [ln for ln in lines if re.match(r"^\d+\s+\[", ln)]
    if not data_lines:
        return {}

    last = data_lines[-1]
    pat = re.compile(r"^(\d+)\s+(\[[^\]]+\])\s+([\deE+\-.]+)\s+(\[[^\]]+\])\s+(\[[^\]]+\])\s+([\deE+\-.]+)\s+(\d+)\s+([\deE+\-.]+)$")
    m = pat.match(last)
    if not m:
        return {}

    phi_d = [float(x) for x in ast.literal_eval(m.group(4))]
    return {
        "phi_d_gravity": phi_d[0],
        "phi_d_magnetic": phi_d[1],
    }


def extract_target_pct(model_root: Path, target_geo_ids=(4, 5)) -> dict:
    """Target volume percentage in active voxels based on geo_id_3d.npy."""
    gpath = model_root / "geology_models" / "geo_id_3d.npy"
    if not gpath.exists():
        return {}

    geo = np.load(gpath).astype(int)
    gvals, gcounts = np.unique(geo, return_counts=True)
    geo_count = {int(v): int(c) for v, c in zip(gvals, gcounts)}

    active_voxels = int(sum(c for v, c in geo_count.items() if v > 0))
    target_voxels = int(sum(geo_count.get(int(gid), 0) for gid in target_geo_ids))
    target_pct = (100.0 * target_voxels / active_voxels) if active_voxels > 0 else np.nan

    return {"target_pct": target_pct}


def extract_report_words(model_root: Path) -> dict:
    """Count words in the latest markdown report."""
    report_dir = model_root / "reports"
    md_path = _latest_file(report_dir, "*.md")
    if not md_path:
        return {}

    txt = md_path.read_text(encoding="utf-8", errors="ignore")
    words = re.findall(r"[A-Za-z]+(?:'[A-Za-z]+)?", txt)
    return {"report_words": len(words)}


## Block 2: Build and Export Unified Metrics Table

Compute per-model metrics, assemble the comparison table, and save the CSV output.


In [2]:
# -------------------------
# Block 2: Build and Export Unified Metrics Table
# -------------------------
rows = []

for model_name, model_root in MODEL_DIRS.items():
    row = {"model": model_name}

    latest_log = _latest_file(model_root / "iteration_model", "Output_*.txt")
    row.update(parse_data_misfit(latest_log))
    row.update(extract_target_pct(model_root, target_geo_ids=(4, 5)))
    row.update(extract_report_words(model_root))

    rows.append(row)

df_table = pd.DataFrame(rows).sort_values("model").reset_index(drop=True)

table_cols = [
    "model",
    "phi_d_gravity",
    "phi_d_magnetic",
    "target_pct",
    "report_words",
]
df_table = df_table[[c for c in table_cols if c in df_table.columns]]

print("Unified comparison table:")
print(df_table.to_string(index=False))

out_table = ROOT / "Hannah_model_metrics_table.csv"
df_table.to_csv(out_table, index=False)
print(f"Saved: {out_table.relative_to(ROOT)}")

Unified comparison table:
 model  phi_d_gravity  phi_d_magnetic  target_pct  report_words
Claude          673.6          1853.0    4.886667          5786
   GPT          667.0          1865.0    3.780000          4343
Gemini          705.7          2166.0    4.661667          2550
  QWEN          624.2          1928.0    5.347500          3271
Saved: Hannah_model_metrics_table.csv
